# Toy walkthrough: spectrale SIS-drempel op een mini-netwerk

Dit notebook is een **didactische versie** van het grp-SIS-onderzoeksproject (*Testing a spectral epidemic threshold for SIS on networks under heavy-tailed recovery*). Alle stappen worden doorlopen met **kleine getallen** die je ook met pen en papier kunt nakijken.

### Onderzoeksvraag in één zin

> Klopt de spectrale drempel $\tau_c \approx 1/\lambda_{\max}$ (en de bijbehorende $\beta_{\mathrm{pred}}$) nog als we recovery-tijden **zwaar-staartig** maken, terwijl het **gemiddelde** $\mathbb{E}[W]$ gelijk blijft?

### Pijplijn (toy ↔ echt project)

| Stap | Wat je doet | Toy (dit notebook) | Echt project |
|------|-------------|-------------------|--------------|
| 1 | Netwerk | 6-cyclus | Erdős–Rényi, $N{=}2000$, $\langle k\rangle{\approx}6$ |
| 2 | $\lambda_{\max}$ | cosinusformule + NumPy | Python op geëxporteerde edge-CSV |
| 3 | Voorspelling | $\beta_{\mathrm{pred}}=1/(\lambda_{\max}\mathbb{E}[W])$ | idem, $\lambda_{\max}{\approx}7{,}18$ |
| 4 | Herstelwet | exp. / power-law / lognormal | zelfde drie regimes in NetLogo |
| 5 | Simulaties | vaste toy-tabel (6 reps) | BehaviorSpace, 24 reps, 20 $\beta$-waarden |
| 6–7 | Drempel | $\hat\beta_{\mathrm{surv}\,50}$ | idem + bootstrap CI |
| 8 | RQ3 | extinctietijden | median tick $\mid$ extinct |

**Geen NetLogo nodig** voor de kern. De analyse gebruikt dezelfde functies als het echte project: `scripts/threshold_estimators.py` en `scripts/analysis_utils.py`.

Lees bij elke stap eerst de tekst, reken desgewenst handmatig mee, en run daarna de code-cel.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def _project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "threshold_estimators.py").is_file():
            return p
    raise FileNotFoundError(
        "Open de repo-root in Cursor (map met scripts/threshold_estimators.py)."
    )


ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmark.recovery import draw_recovery_time
from scripts.analysis_utils import degree_moments_from_endpoints, largest_adjacency_eigenvalue
from scripts.threshold_estimators import (
    bootstrap_survival_threshold,
    row_threshold_summary,
    survival_curve,
    threshold_smallest_beta,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
ROOT

---
## Stap 1 — Klein contactnetwerk

### Context

In het SIS-model (susceptible–infected–susceptible) kan een agent na herstel **opnieuw** besmet raken. Infectie verspreidt zich langs **contacten** (randen in een netwerk). De vraag is: bij welke infectiekracht $\beta$ sterft een uitbraak uit, en wanneer blijft ze hangen?

In het echte project is het netwerk een **Erdős–Rényi**-graf met ~2000 knopen. Hier gebruiken we een **6-cyclus** (ring): elk knoop heeft precies 2 buren. Dat is klein genoeg om alles met de hand te volgen, maar de analyse-stappen zijn identiek.

```
0 — 1 — 2
|       |
5 — 4 — 3
```

### Vaste parameters (toy vs. project)

| Parameter | Betekenis | Toy | Echt project |
|-----------|-----------|-----|--------------|
| $N$ | aantal agents | 6 | 2000 |
| $\mathbb{E}[W]$ | gem. hersteltijd (ticks) | 5 | 5 |
| $I_0$ | initieel besmet | 1 | 5 |
| max-ticks | simulatie-horizon | 500 | 10 000 |
| $\beta$ | infectiekans per tick langs een link | wordt gevarieerd | sweep 0.018–0.056 |

### Wat de code doet

Bouwt de lijst met randen, de **adjacency-matrix** $A$ ($A_{ij}=1$ als $i$ en $j$ contact hebben) en toont die als tabel.

In [ ]:
N_NODES = 6
RECOVERY_MEAN = 5.0
MAX_TICKS = 500
INITIAL_INFECTED = 1

# Randen van de 6-cyclus: 0—1—2—3—4—5—0
edges = [(i, (i + 1) % N_NODES) for i in range(N_NODES)]
a = np.array([u for u, _ in edges], dtype=np.int64)
b = np.array([v for _, v in edges], dtype=np.int64)

adj = np.zeros((N_NODES, N_NODES), dtype=float)
adj[a, b] = 1.0
adj[b, a] = 1.0

print("Randen:", edges)
print(f"Aantal randen m = {len(edges)}  →  gemiddelde graad ⟨k⟩ = {2*len(edges)/N_NODES:.0f}")
print("\nAdjacency-matrix A:")
display(pd.DataFrame(adj, index=range(N_NODES), columns=range(N_NODES)).astype(int))

### Interpretatie stap 1

- Er zijn **6 randen** en **6 knopen** → $\langle k\rangle = 2m/N = 12/6 = 2$.
- $A$ is **symmetrisch**: contact is wederzijds.
- In NetLogo exporteer je dezelfde informatie als edge-lijst (CSV); hier bouwen we $A$ direct op.

---
## Stap 2 — $\lambda_{\max}$ uit de adjacency-matrix

### Waarom een eigenwaarde?

Netwerk-epidemiologie voorspelt dat de **grootste eigenwaarde** $\lambda_{\max}$ van de adjacency-matrix samenhangt met hoe gemakkelijk infectie door het netwerk kan blijven circuleren. Intuïtie: een hogere $\lambda_{\max}$ betekent meer "samenhang" in het contactpatroon → infectie kan makkelijker blijven hangen.

De spectrale referentie in het project is:

$$\tau_c \approx \frac{c}{\lambda_{\max}}, \quad c=1 \text{ (mean-field benchmark)}$$

### Handmatig voor de ring

Een $N$-cyclus heeft een **circulante** adjacency-matrix. Alle eigenwaarden zijn:

$$\lambda_j = 2\cos\!\left(\frac{2\pi j}{N}\right), \quad j=0,1,\ldots,N-1$$

Voor $N=6$:

| $j$ | $\lambda_j$ |
|-----|----------|
| 0 | $2\cos(0) = \mathbf{2}$ |
| 1 | $2\cos(60°) = 1$ |
| 2 | $2\cos(120°) = -1$ |
| 3 | $-2$ |
| 4 | $-1$ |
| 5 | $1$ |

Dus **$\lambda_{\max} = 2$**. De code controleert dit met `numpy.linalg.eigvalsh`.

### Graden (controle + MF-alternatief)

Naast het spectrum berekenen we $\langle k\rangle$ en $\langle k^2\rangle$ — die gebruiken we later voor een **heterogene mean-field** referentie $\tau_{\mathrm{het}} = \langle k\rangle / \langle k^2\rangle$.

In [ ]:
j = np.arange(N_NODES)
eigs_circulant = 2.0 * np.cos(2.0 * np.pi * j / N_NODES)
lambda_hand = float(eigs_circulant.max())
lambda_numpy = largest_adjacency_eigenvalue(adj)
k_mean, k2_mean, tau_hom, tau_het = degree_moments_from_endpoints(a, b, N_NODES)

print("Alle eigenwaarden (hand):", np.round(eigs_circulant, 4))

summary_spectral = pd.DataFrame(
    {
        "quantity": [
            "lambda_max (hand)",
            "lambda_max (numpy)",
            "<k>",
            "<k^2>",
            "tau_hom = 1/<k>",
            "tau_het = <k>/<k^2>",
        ],
        "value": [lambda_hand, lambda_numpy, k_mean, k2_mean, tau_hom, tau_het],
    }
)
display(summary_spectral)
assert np.isclose(lambda_hand, lambda_numpy), "Hand en NumPy moeten overeenkomen"

### Interpretatie stap 2

- Op deze **reguliere** ring valt $\tau_{\mathrm{hom}} = 1/\langle k\rangle = 0{,}5$ samen met $\tau_{\mathrm{pred}} = 1/\lambda_{\max} = 0{,}5$.
- Op jullie ER-graaf (onregelmatige graden) schelen spectrale en MF-referenties iets (~2,6%); de empirische drempel ligt daar **veel** hoger dan beide.

---
## Stap 3 — Spectrale voorspelling $\beta_{\mathrm{pred}}$ en $\tau_{\mathrm{pred}}$

### Effectieve transmissie

Het project koppelt de discrete NetLogo-parameter $\beta$ (infectiekans per tick langs een link) aan een continue-theorie-schaal:

$$\tau = \beta \cdot \mathbb{E}[W]$$

Hier is $\mathbb{E}[W]$ de **gemiddelde** tijd dat iemand besmet blijft (in ticks). Bij vaste $\mathbb{E}[W]=5$ is $\tau$ evenredig met $\beta$.

### Drempel op de $\tau$-as en op de $\beta$-as

$$\tau_{\mathrm{pred}} = \frac{1}{\lambda_{\max}}, \qquad
\beta_{\mathrm{pred}} = \frac{\tau_{\mathrm{pred}}}{\mathbb{E}[W]} = \frac{1}{\lambda_{\max}\,\mathbb{E}[W]}$$

**Invulles voor de toy-ring:** $\lambda_{\max}=2$, $\mathbb{E}[W]=5$ →

- $\tau_{\mathrm{pred}} = 1/2 = 0{,}50$
- $\beta_{\mathrm{pred}} = 0{,}50/5 = \mathbf{0{,}10}$

### Wat betekent dit?

Onder de mean-field benchmark ($c=1$) zou je verwachten dat infectie **rond** $\beta \approx 0{,}10$ de grens tussen uitsterven en persisteren ligt. In het echte project test je of de **empirische** drempel $\hat\beta_{\mathrm{surv}\,50}$ daar dichtbij ligt (**RQ1**, exponentieel herstel) of systematisch verschuift (**RQ2**, zware staarten).

In [ ]:
LAMBDA_MAX = lambda_numpy
tau_pred = 1.0 / LAMBDA_MAX
beta_pred = tau_pred / RECOVERY_MEAN
beta_het = tau_het / RECOVERY_MEAN

print("Handrekening:")
print(f"  tau_pred  = 1 / lambda_max = 1 / {LAMBDA_MAX:.0f} = {tau_pred:.2f}")
print(f"  beta_pred = tau_pred / E[W] = {tau_pred:.2f} / {RECOVERY_MEAN:.0f} = {beta_pred:.2f}")
print(f"\nAlternatief (heterogene MF): beta_het = {beta_het:.2f}")

### Interpretatie stap 3

| Grootheid | Waarde (toy) | Rol |
|-----------|--------------|-----|
| $\tau_{\mathrm{pred}}$ | 0,50 | referentie op $\tau$-as |
| $\beta_{\mathrm{pred}}$ | 0,10 | verticale stippellijn in survival-plots |
| $\beta_{\mathrm{het}}$ | 0,10 | alternatieve MF-lijn (hier gelijk aan spectrale) |

In het rapport vergelijk je later $\hat\beta / \beta_{\mathrm{pred}}$ — een ratio $>1$ betekent: de simulatie heeft **meer** $\beta$ nodig dan de theorie voorspelt (conservatieve spectrale lijn).

---
## Stap 4 — Hersteltijdverdelingen (grp-SIS)

### Wat varieert en wat blijft vast?

Het **grp-SIS**-kader (Tang et al.) houdt de SIS-structuur (herstel → weer susceptible), maar laat de **vorm** van de hersteltijd $W$ vrij. In het project vergelijk je drie wetten bij **dezelfde** $\mathbb{E}[W]=5$:

| Regime | Karakter | Effect op dynamiek |
|--------|----------|-------------------|
| **Exponentieel** | geheugenloos (Markoviaans) | baseline, sluit aan bij klassieke theorie |
| **Power-law (Tang)** | zware rechterstaart | sommigen blijven lang besmet |
| **Lognormal (Tang)** | brede spreiding | ook langdurige besmettingen, andere vorm dan power-law |

### Belangrijk voor drempelvergelijking

Omdat $\tau = \beta \cdot \mathbb{E}[W]$ en $\mathbb{E}[W]$ **vast** is, verandert de **theoretische** $\beta_{\mathrm{pred}}$ niet tussen regimes. Eventuele verschuivingen in $\hat\beta_{\mathrm{surv}\,50}$ komen dus door de **dynamiek**, niet doordat je het gemiddelde herstel verschuift.

De histogrammen hieronder tonen: zelfde gemiddelde, **verschillende staarten** — precies het experimentele contrast van RQ2.

In [ ]:
rng = np.random.default_rng(42)
n_draws = 10_000
regimes = ["exponential", "power_law_tang", "lognormal_tang"]
draws = {
    r: [draw_recovery_time(r, RECOVERY_MEAN, rng) for _ in range(n_draws)] for r in regimes
}
stats = pd.DataFrame(
    {
        "regime": regimes,
        "mean": [np.mean(draws[r]) for r in regimes],
        "median": [np.median(draws[r]) for r in regimes],
        "p95": [np.percentile(draws[r], 95) for r in regimes],
        "max": [np.max(draws[r]) for r in regimes],
    }
)
display(stats.round(2))

fig, ax = plt.subplots()
for r in regimes:
    ax.hist(draws[r], bins=50, alpha=0.45, density=True, label=r.replace("_", " "))
ax.set_xlabel("Recovery time W (ticks)")
ax.set_ylabel("Density")
ax.set_title(f"Recovery draws at E[W]={RECOVERY_MEAN:.0f}")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretatie stap 4

- **Mediaan** $\neq$ **gemiddelde** bij zware staarten: veel korte hersteltijden + enkele zeer lange.
- Bij infectie trekt elke agent **een eigen** $W$; de populatie is een mix van "kort besmet" en "lang besmet".
- Hypothese project: zware staarten → **makkelijker persisteren**. In het rapport blijkt het beeld genuanceerder (power-law iets lager, lognormal iets hoger dan exponentieel).

---
## Stap 5 — Simulatie-uitkomsten (toy BehaviorSpace)

### Wat het echte project logt

BehaviorSpace draait het NetLogo-model vele keren en schrijft per run o.a.:

| Output | Betekenis |
|--------|-----------|
| `bs-out-extinct` | 1 = alle agents genezen vóór max-ticks; 0 = nog besmet op horizon |
| `bs-out-final-tick` | tijdstip van uitsterven (of max-ticks bij survival) |
| `bs-out-late-mean-prevalence` | gem. prevalentie in laat venster (alleen bij survival) |

### Toy-data (vast, handmatig na te rekenen)

Hier gebruiken we een **vooraf ingevulde** tabel met **6 replicates** per $(\beta, \text{regime})$ — genoeg om survival-percentages als breuken te tellen (bijv. 4/6 = 67%).

**Definities:**
- `bs_out_extinct = 1` → run is **uitgestorven**
- `bs_out_extinct = 0` → run **overleeft** de horizon (tick 500)

### Oefening vóór je de code runt

Neem regime **exponentieel**, $\beta = 0{,}12$. De zes uitkomsten zijn: overleefd, overleefd, uitgestorven (tick 52), overleefd, overleefd, uitgestorven (tick 45).

→ Survival rate = ? (antwoord onderaan notebook)

In [ ]:
BETA_GRID = [0.06, 0.08, 0.10, 0.12, 0.14]
N_REP = 6

# Per regime: dict beta -> list of (extinct_flag, final_tick_if_extinct_or_MAX)
TOY_OUTCOMES = {
    "exponential": {
        0.06: [(1, 120), (1, 95), (1, 88), (1, 102), (1, 77), (1, 110)],
        0.08: [(1, 65), (1, 72), (1, 58), (1, 81), (1, 69), (0, MAX_TICKS)],
        0.10: [(1, 55), (1, 48), (1, 62), (0, MAX_TICKS), (1, 51), (0, MAX_TICKS)],
        0.12: [(0, MAX_TICKS), (0, MAX_TICKS), (1, 52), (0, MAX_TICKS), (0, MAX_TICKS), (1, 45)],
        0.14: [(0, MAX_TICKS)] * N_REP,
    },
    "power_law_tang": {
        0.06: [(1, 90)] * N_REP,
        0.08: [(1, 42), (0, MAX_TICKS), (1, 38), (0, MAX_TICKS), (1, 55), (1, 47)],
        0.10: [(0, MAX_TICKS), (1, 35), (0, MAX_TICKS), (0, MAX_TICKS), (1, 41), (1, 33)],
        0.12: [(0, MAX_TICKS), (0, MAX_TICKS), (0, MAX_TICKS), (0, MAX_TICKS), (1, 28), (0, MAX_TICKS)],
        0.14: [(0, MAX_TICKS)] * N_REP,
    },
    "lognormal_tang": {
        0.06: [(1, 70)] * N_REP,
        0.08: [(1, 60)] * N_REP,
        0.10: [(1, 50), (1, 44), (1, 58), (1, 52), (1, 46), (0, MAX_TICKS)],
        0.12: [(1, 30), (1, 25), (0, MAX_TICKS), (1, 28), (0, MAX_TICKS), (1, 22)],
        0.14: [(0, MAX_TICKS), (0, MAX_TICKS), (1, 20), (0, MAX_TICKS), (0, MAX_TICKS), (1, 18)],
    },
}

rows = []
for regime, by_beta in TOY_OUTCOMES.items():
    for beta, outcomes in by_beta.items():
        for rep, (ext, tick) in enumerate(outcomes, start=1):
            rows.append(
                {
                    "recovery_regime": regime,
                    "infection_prob": beta,
                    "replicate": rep,
                    "bs_out_extinct": ext,
                    "bs_out_final_tick": tick,
                }
            )

toy_df = pd.DataFrame(rows)
print(f"Totaal {len(toy_df)} runs = {len(regimes)} regimes × {len(BETA_GRID)} beta × {N_REP} reps")
toy_df.head(12)

---
## Stap 6 & 7 — Survival curves en empirische drempel

### Survival rate

Voor elke $\beta$ op het grid:

$$\text{survival rate}(\beta) = \frac{\#\{\text{runs met } \texttt{extinct}=0\}}{\#\text{replicates}}$$

Dit is precies wat `survival_curve()` in `scripts/threshold_estimators.py` berekent.

### Empirische drempel $\hat\beta_{\mathrm{surv}\,50}$

> **Kleinste** $\beta$ op het grid waarvoor survival rate $\ge 50\%$.

Voorbeeld (exponentieel): bij $\beta=0{,}10$ is survival 2/6 = 33% (< 50%); bij $\beta=0{,}12$ is het 4/6 = 67% (≥ 50%) → $\hat\beta_{\mathrm{surv}\,50} = 0{,}12$.

### Koppeling aan onderzoeksvragen

| Vraag | Wat je vergelijkt |
|-------|------------------|
| **RQ1** | $\hat\beta_{\mathrm{surv}\,50}$ (exponentieel) vs. $\beta_{\mathrm{pred}}$ — klopt de spectrale lijn? |
| **RQ2** | Verschuift $\hat\beta_{\mathrm{surv}\,50}$ bij power-law / lognormal t.o.v. exponentieel? |
| **RQ3** | Zijn verschillen vooral in **drempel** of in **extinctietijd** / prevalentie? (stap 8) |

### Bootstrap (kort)

Met 6 replicates is één survival-rate al een **steekproef**. De bootstrap (400× resamplen met teruglegging **per** $\beta$) schat de onzekerheid in $\hat\beta_{\mathrm{surv}\,50}$. In het echte project: 24 replicates → smallere CI's.

In [ ]:
threshold_rows = []
curve_by_regime = {}

for regime in TOY_OUTCOMES:
    sub = toy_df[toy_df["recovery_regime"] == regime].copy()
    curve = survival_curve(
        sub,
        inf_col="infection_prob",
        ext_col="bs_out_extinct",
        tick_col="bs_out_final_tick",
    )
    curve_by_regime[regime] = curve
    beta_hat = threshold_smallest_beta(curve, "infection_prob", "p_survive", q=0.5)
    med, lo, hi = bootstrap_survival_threshold(
        sub, "infection_prob", "bs_out_extinct", q=0.5, n_boot=400, rng=np.random.default_rng(1)
    )
    summary = row_threshold_summary(
        sub,
        "infection_prob",
        "bs_out_extinct",
        prev_col=None,
        tick_col="bs_out_final_tick",
        beta_pred=beta_pred,
        n_boot=400,
    )
    threshold_rows.append(
        {
            "regime": regime,
            "beta_hat_surv_50": beta_hat,
            "tau_hat_50": beta_hat * RECOVERY_MEAN,
            "ratio_over_pred": beta_hat / beta_pred,
            "tau_ratio": (beta_hat * RECOVERY_MEAN) / tau_pred,
            "boot_median": med,
            "boot_ci_low": lo,
            "boot_ci_high": hi,
            "median_tick_if_extinct": summary.get("median_tick_if_extinct"),
        }
    )

threshold_table = pd.DataFrame(threshold_rows).round(4)
display(threshold_table)

print("\nSurvival per beta (handmatig nakijken):")
for regime, curve in curve_by_regime.items():
    print(f"\n=== {regime} ===")
    display(curve.assign(p_survive_pct=lambda d: (100 * d["p_survive"]).round(1)))

### Interpretatie stap 6 & 7 (verwachte toy-resultaten)

| Regime | $\hat\beta_{\mathrm{surv}\,50}$ | $\hat\beta / \beta_{\mathrm{pred}}$ | $\hat\tau_{50}$ |
|--------|------------------------------|----------------------------------|----------------|
| Exponentieel | 0,12 | 1,20 | 0,60 |
| Power-law | 0,10 | 1,00 | 0,50 |
| Lognormal | 0,14 | 1,40 | 0,70 |

**RQ1:** exponentieel ligt **boven** $\beta_{\mathrm{pred}}$ (ratio 1,2) — de spectrale lijn is een **conservatieve** ondergrens.

**RQ2:** power-law **lager** dan exponentieel (0,10 vs 0,12); lognormal **hoger** (0,14). Zware staarten verschuiven de drempel, maar niet allemaal dezelfde kant op.

**Let op:** met een grof $\beta$-grid (stap 0,02) zit de ware drempel *tussen* gridpunten; in het rapport is $\Delta\beta = 0{,}002$.

In [ ]:
fig, ax = plt.subplots()
labels = {
    "exponential": "Exponential",
    "power_law_tang": "Power law (Tang)",
    "lognormal_tang": "Lognormal (Tang)",
}
for regime, curve in curve_by_regime.items():
    ax.plot(curve["infection_prob"], curve["p_survive"], "o-", label=labels[regime])
ax.axvline(beta_pred, color="k", ls="--", lw=1.2, label=rf"$\beta_{{pred}}$={beta_pred:.2f}")
ax.axhline(0.5, color="gray", ls=":", lw=1.2, label="50% survival")
ax.set_xlabel(r"Infection probability $\beta$")
ax.set_ylabel("Survival rate (not extinct at horizon)")
ax.set_title("Toy 6-cycle: survival vs $\beta$")
ax.legend()
plt.tight_layout()
plt.show()

### De figuur lezen

- **Horizontale stippellijn (50%):** definitie van $\hat\beta_{\mathrm{surv}\,50}$.
- **Verticale streep ($\beta_{\mathrm{pred}}$):** spectrale voorspelling — alle drie de curves zitten **rechts** van die lijn: je hebt meer $\beta$ nodig dan de theorie zegt.
- **Verticale verschuiving tussen curves:** RQ2 — power-law curve snijdt 50% het eerst (links), lognormal het laatst (rechts).
- In het echte project zitten alle curves nog verder rechts van $\beta_{\mathrm{pred}} \approx 0{,}028$.

---
## Stap 8 — Extinctietijden (RQ3)

### Waarom apart van de drempel?

**RQ3** vraagt of zware-staart-herstel vooral de **drempel** verschuift, of vooral **hoe lang** uitbraken duren en **hoe variabel** prevalentie is.

De drempel $\hat\beta_{\mathrm{surv}\,50}$ kijkt alleen naar **binaire** survival op de horizon. Extinctietijden vertellen iets over **transiënten**: als een run wél uitsterft, hoe snel gebeurt dat?

We bekijken hier **uitgestorven** runs bij $\beta = 0{,}12$ en nemen de **mediaan** van `bs_out_final_tick`.

In [ ]:
beta_focus = 0.12
ext_rows = []
for regime in TOY_OUTCOMES:
    sub = toy_df[
        (toy_df["recovery_regime"] == regime)
        & (toy_df["infection_prob"] == beta_focus)
        & (toy_df["bs_out_extinct"] >= 0.5)
    ]
    ticks = sub["bs_out_final_tick"].to_numpy()
    ext_rows.append(
        {
            "regime": regime,
            "beta": beta_focus,
            "n_extinct": len(ticks),
            "extinction_ticks": ", ".join(map(str, ticks)),
            "median_tick": float(np.median(ticks)) if len(ticks) else np.nan,
        }
    )
display(pd.DataFrame(ext_rows))

### Interpretatie stap 8

Bij $\beta = 0{,}12$ (toy):

| Regime | Uitgestorven runs | Extinctieticks | Mediaan |
|--------|-------------------|----------------|--------|
| Exponentieel | 2 | 52, 45 | **48,5** |
| Power-law | 1 | 28 | **28** |
| Lognormal | 4 | 30, 25, 28, 22 | **27** |

Als een uitbraak uitsterft, gebeurt dat onder zware-staart-herstel **sneller** dan onder exponentieel — terwijl power-law bij *lagere* $\beta$ juist **vaker** overleeft. Drempel en extinctietijd zijn dus **verschillende** meetinstrumenten (zoals in het rapport: median extinct tick 22 vs 17 vs 15 op de ER-graaf).

---
## Vergelijking met het echte project

De **logica** is hetzelfde; de **schaal** niet. Onderstaande waarden komen uit `report/generated_quantities.tex` (ER, seed 10001).

| | Toy ring | Echt ER |
|--|----------|--------|
| $N$ | 6 | 2000 |
| $\lambda_{\max}$ | 2 | 7,18 |
| $\beta_{\mathrm{pred}}$ | 0,10 | 0,028 |
| $\hat\beta$ exponentieel | 0,12 (ratio 1,20) | 0,046 (ratio 1,65) |
| $\hat\beta$ power-law | 0,10 (ratio 1,00) | 0,044 (ratio 1,58) |
| $\hat\beta$ lognormal | 0,14 (ratio 1,40) | 0,050 (ratio 1,80) |

**Wat blijft gelijk:** empirische drempel boven spectrale lijn; power-law laagste, lognormal hoogste; RQ3-signalen in extinctietijden.

**Wat verschilt:** absolute ratios (~1,2–1,4 vs ~1,6–1,8) door $N$, discrete tijd, en gridfijnheid.

In [ ]:
real = pd.DataFrame(
    {
        "setting": ["toy ring", "real ER"],
        "N": [6, 2000],
        "lambda_max": [2.0, 7.181],
        "beta_pred": [0.10, 0.02785],
        "beta_hat_exp": [0.12, 0.046],
        "beta_hat_power": [0.10, 0.044],
        "beta_hat_logn": [0.14, 0.050],
        "ratio_exp": [1.20, 1.652],
    }
)
display(real)

---
## Inference-niveaus (Wieringa / IM1312)

Het project doorloopt bewust verschillende **inferentieniveaus**. Vermijd ze door elkaar te halen:

| Niveau | Vraag | Voorbeeld in dit notebook |
|--------|-------|---------------------------|
| **(a) Descriptief** | Wat gebeurde in onze runs? | "Bij $\beta=0{,}12$ overleefden 4/6 exponentiële runs" |
| **(d) Statistisch** | Generaliseert dit? | Bootstrap CI op $\hat\beta$; meerdere ER-seeds in het rapport |
| **(b) Abductief** | Wat is een plausibele verklaring? | "Gap met $\beta_{\mathrm{pred}}$ door discrete tijd + eindige $N$" |
| **(c) Analogisch** | Geldt dit elders? | "Verwacht vergelijkbare shift op lattice/ring" (nog testen) |

**Fout:** *"Power-law recovery **veroorzaakt** 20% lagere drempels."* (causaal, niet bewezen)

**Goed:** *"Power-law $\hat\beta$ lag 0,02 onder exponentieel op dezelfde toy-graaf; consistent met langere infectie-staarten, maar $N=6$ en grof grid beperken de precisie."*

---
## Optioneel — Mini-simulatie op de ring

De vaste toy-tabel is **deterministisch** voor onderwijs. Hieronder staat een eenvoudige discrete-time SIS op dezelfde 6-cyclus — dichter bij NetLogo, maar met **stochastische** uitkomsten.

Zet `RUN_MINI_SIM = True` om te draaien. Verwacht dat $\hat\beta_{\mathrm{surv}\,50}$ **afwijkt** van de vaste tabel: dat illustreert waarom het echte project 24 replicates gebruikt.

In [ ]:
def simulate_sis_ring(
    beta: float,
    regime: str,
    *,
    rng: np.random.Generator,
    max_ticks: int = MAX_TICKS,
    initial_infected: int = INITIAL_INFECTED,
) -> tuple[int, int]:
    """Return (extinct_flag, final_tick). extinct=1 if I(t)=0 before max_ticks."""
    infected = np.zeros(N_NODES, dtype=bool)
    start_nodes = rng.choice(N_NODES, size=initial_infected, replace=False)
    infected[start_nodes] = True
    recovery_deadline = np.zeros(N_NODES, dtype=int)
    for node in start_nodes:
        recovery_deadline[node] = draw_recovery_time(regime, RECOVERY_MEAN, rng)

    for t in range(1, max_ticks + 1):
        recover_now = infected & (recovery_deadline <= t)
        infected[recover_now] = False
        if not infected.any():
            return 1, t
        new_inf = infected.copy()
        for u, v in edges:
            if infected[u] and not infected[v] and rng.random() < beta:
                new_inf[v] = True
                recovery_deadline[v] = t + draw_recovery_time(regime, RECOVERY_MEAN, rng)
            if infected[v] and not infected[u] and rng.random() < beta:
                new_inf[u] = True
                recovery_deadline[u] = t + draw_recovery_time(regime, RECOVERY_MEAN, rng)
        infected = new_inf
    return 0, max_ticks


RUN_MINI_SIM = False  # zet op True om te draaien

if RUN_MINI_SIM:
    sim_rng = np.random.default_rng(2026)
    sim_rows = []
    for regime in regimes:
        for beta in BETA_GRID:
            for rep in range(N_REP):
                ext, tick = simulate_sis_ring(beta, regime, rng=sim_rng)
                sim_rows.append(
                    {
                        "recovery_regime": regime,
                        "infection_prob": beta,
                        "replicate": rep + 1,
                        "bs_out_extinct": ext,
                        "bs_out_final_tick": tick,
                    }
                )
    sim_df = pd.DataFrame(sim_rows)
    for regime in regimes:
        sub = sim_df[sim_df["recovery_regime"] == regime]
        curve = survival_curve(sub, "infection_prob", "bs_out_extinct")
        beta_hat = threshold_smallest_beta(curve, "infection_prob", "p_survive", q=0.5)
        print(f"{regime}: beta_hat_surv_50 = {beta_hat:.3f} (ratio {beta_hat/beta_pred:.2f})")
else:
    print("Zet RUN_MINI_SIM = True om de optionele simulatie te draaien.")

---
## Oefeningen en antwoordsleutel

### Opdrachten (pen en papier)

1. Bereken alle zes eigenwaarden van de 6-cyclus en bevestig $\lambda_{\max}=2$.
2. Bereken $\beta_{\mathrm{pred}}$ met $\mathbb{E}[W]=5$.
3. Exponentieel, $\beta=0{,}12$: hoeveel van de 6 runs hebben `extinct=0`? Wat is de survival rate?
4. Bepaal $\hat\beta_{\mathrm{surv}\,50}$ voor elk van de drie regimes.
5. Bereken $\hat\tau_{50} = \hat\beta_{\mathrm{surv}\,50} \cdot 5$ en de ratio $\hat\tau_{50}/\tau_{\mathrm{pred}}$ voor exponentieel.
6. Bij $\beta=0{,}12$: mediaan extinctietick voor exponentieel (alleen uitgestorven runs).
7. Welke conclusies uit het toy-voorbeeld blijven gelijk in het echte ER-project?

### Antwoordsleutel

| # | Antwoord |
|---|----------|
| 1 | $\lambda \in \{2, 1, -1, -2, -1, 1\}$ → max = 2 |
| 2 | $\beta_{\mathrm{pred}} = 1/(2 \times 5) = 0{,}10$ |
| 3 | 4 overleefd → survival = 4/6 ≈ **67%** |
| 4 | exp. **0,12**; power-law **0,10**; lognormal **0,14** |
| 5 | $\hat\tau_{50} = 0{,}60$; ratio $0{,}60/0{,}50 = \mathbf{1{,}20}$ |
| 6 | ticks 52 en 45 → mediaan $(52+45)/2 =$ **48,5** |
| 7 | Drempel boven spectrale lijn; power-law laagste, lognormal hoogste; extinctietijden onderscheiden regimes sterker dan kleine drempelverschillen |